In [ ]:
!pip install pycryptodome ecdsa base58 tqdm

import hashlib
import binascii
from Crypto.Hash import SHA256, RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import random
from tqdm import tqdm

TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
TARGET_HASH160 = "a0b0d60e5991578ed37cbda2b17d8b2ce23ab295"
SAMPLE_PRIVS = [
    "64cef70550c928c7244bc04bafb3a713fed1850c34382db9a7df9ad66b5b2847",  # 1Fe6BX…
    "94c9b3e5aeab47bf2de5dc118de9c43ad36bd96c71513101420d9a062e46122f",  # 1FeWcM…
    "7eb0f34f756667d0643347fda75c719ed9f424190dfc057deacc1da1338f4b0e",  # 1FerV3…
    "c9b1477d4ef53febd524b09273ee41cd769140638df7b8c8227e260ee1a539bd",  # 1FeUMq…
    "70ae2cf1a380149dbdf6db17152ca587dcdae9176adbeba723b972a04f0c1cc6"   # 1Fe3ZY…
]

def privkey_to_address(priv_hex):
    try:
        priv_bytes = binascii.unhexlify(priv_hex)
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = SHA256.new(pub).digest()
        ripemd = RIPEMD160.new(sha).digest()
        payload = b'\x00' + ripemd
        checksum = SHA256.new(SHA256.new(payload).digest()).digest()[:4]
        return base58.b58encode(payload + checksum).decode(), ripemd.hex()
    except:
        return None, None

def hamming_distance(s1, s2):
    len_diff = abs(len(s1) - len(s2))
    return sum(c1 != c2 for c1, c2 in zip(s1, s2)) + len_diff

def flip_bit(priv_hex, position):
    priv_bytes = bytearray(binascii.unhexlify(priv_hex))
    byte_idx = position // 8
    bit_idx = position % 8
    if byte_idx < len(priv_bytes):
        priv_bytes[byte_idx] ^= (1 << bit_idx)
    return binascii.hexlify(priv_bytes).decode('utf-8')

# Step 1: Verify Hamming
def verify_samples():
    print("Verifying Sample Privkeys\n")
    closest_priv = None
    closest_hamm = float('inf')
    closest_addr = None

    for priv in SAMPLE_PRIVS:
        addr, hash160 = privkey_to_address(priv)
        if not addr:
            continue
        addr_hamm = hamming_distance(addr, TARGET_ADDRESS)
        hash160_hamm = hamming_distance(hash160, TARGET_HASH160)
        print(f"Priv: {priv}")
        print(f"Addr: {addr}, Addr Hamming: {addr_hamm}")
        print(f"Hash160: {hash160}, Hash160 Hamming: {hash160_hamm}\n")
        if hash160_hamm < closest_hamm:
            closest_priv = priv
            closest_hamm = hash160_hamm
            closest_addr = addr

    print(f"Closest - Priv: {closest_priv}, Addr: {closest_addr}, Hash160 Hamming: {closest_hamm}")
    return closest_priv, closest_hamm, closest_addr

# Step 2: Bitflip Closest
def bitflip_closest(base_priv, sample_size=1000000):
    best_priv = None
    best_hamm = float('inf')
    best_addr = None

    base_addr, base_hash160 = privkey_to_address(base_priv)
    base_hamm = hamming_distance(base_hash160, TARGET_HASH160)
    print(f"\nBitflipping Closest - Priv: {base_priv}, Addr: {base_addr}, Hash160 Hamming: {base_hamm}")

    for _ in tqdm(range(sample_size), desc="Bitflip"):
        num_flips = random.randint(1, 16)
        flip_positions = random.sample(range(256), num_flips)

        priv_bytes = bytearray(binascii.unhexlify(base_priv))
        for pos in flip_positions:
            byte_idx = pos // 8
            bit_idx = pos % 8
            priv_bytes[byte_idx] ^= (1 << bit_idx)
        flipped_priv = binascii.hexlify(priv_bytes).decode('utf-8')

        addr, hash160 = privkey_to_address(flipped_priv)
        if not addr:
            continue

        hamm = hamming_distance(hash160, TARGET_HASH160)
        if hamm < best_hamm:
            best_priv = flipped_priv
            best_hamm = hamm
            best_addr = addr
            print(f"New Best - Priv: {best_priv}, Addr: {best_addr}, Hash160 Hamming: {hamm}")
        if hamm == 0:
            print(f"Exact Match! Priv: {best_priv}, Addr: {best_addr}")
            return best_priv, 0, best_addr

    print(f"\nResults:")
    print(f"Best Priv: {best_priv}")
    print(f"Best Addr: {best_addr}")
    print(f"Best Hamming: {best_hamm}")
    return best_priv, best_hamm, best_addr

# Run It
closest_priv, closest_hamm, closest_addr = verify_samples()
if closest_hamm > 0:
    best_priv, best_hamm, best_addr = bitflip_closest(closest_priv)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 9.3 MB/s eta 0:00:00
Verifying Sample Privkeys

Priv: 64cef70550c928c7244bc04bafb3a713fed1850c34382db9a7df9ad66b5b2847
Addr: 1Fe6BX457sT1QDc3i8rwaWx7ogUAsoAZvA, Addr Hamming: 31
Hash160: a0957a0afcd6931a60ac1d431f053653b7d92e2e, Hash160 Hamming: 35

Priv: 94c9b3e5aeab47bf2de5dc118de9c43ad36bd96c71513101420d9a062e46122f
Addr: 1FeWcMsBjvVjBBCpFuDqb7JukunWiiFNJ2, Addr Hamming: 30
Hash160: a0a9de52cb439d5105aa664bc5174d222a7cb670, Hash160 Hamming: 35

Priv: 7eb0f34f756667d0643347fda75c719ed9f424190dfc057deacc1da1338f4b0e
Addr: 1FerV33Yyy11YU6wYbHnTHCJKgfGzifpiY, Addr Hamming: 31
Hash160: a0ba754c236adb589d9a94d1e8fc507546094bdc, Hash160 Hamming: 37

Priv: c9b1477d4ef53febd524b09273ee41cd769140638df7b8c8227e260ee1a539bd
Addr: 1FeUMq11YbCnX9rGfLyQ6YHQ21j6s13Fcu, Addr Hamming: 31
Hash160: a0a7fd62649a9eb591e18e6f7b2d4f3231e4022d, Hash160 Hamming: 35

Priv: 70

Bitflip:   0%|          | 73/1000000 [00:00<23:01, 723.71it/s]

New Best - Priv: 64cef70550d928c7244b804befb3a713fed1850c35382db9a7df9ad66b5b2847, Addr: 1G1kvciF5yxc3oCAzrwRpKwJQTRdmvBdrk, Hash160 Hamming: 38
New Best - Priv: 64ceb70550c928c7244bc04bafb3a713fed1850c34382db9a7df9ad66b5b2847, Addr: 18aF1KfEMhViZjPpVoSjmoWh1P7nJJSFjg, Hash160 Hamming: 37
New Best - Priv: 64cef78550c928c7344bc06ba7a3a613fed1870c34382db9a7df9ad6695b2845, Addr: 1BzgqJhb4TZ4Gu2bCwPaNfxy4oU3qALVbM, Hash160 Hamming: 35
New Best - Priv: 64cff705584928c7a44bc04b8fb3a7137ed0850c34382db927cf9ad66b5b2807, Addr: 19fFfCVXY8soTXQTnx4JqwuqRQvBPLA7bt, Hash160 Hamming: 34
New Best - Priv: 6cceb70550c92ac7264bc069afb3a713fed1850c34382db9a3df9ed66b1b2847, Addr: 1X6SWXkYS3ghU9RJmaLuzKmnYuUcaxdmS, Hash160 Hamming: 33


Bitflip:   0%|          | 612/1000000 [00:00<21:44, 765.96it/s]

New Best - Priv: 64cef70510c928cf244bc04fafb3a713fed1850c34282dbdb7df9ad66b5b2847, Addr: 12TtGYx4VWgS7cZSNqTfkX67u3oaMK6ACR, Hash160 Hamming: 32


...

New Best - Priv: 64c6f70550c928c7244bc04b8bb327b3fe91850c30382fb9a7df1ad64b7b0847, Addr: 1N6jrgkQTUTd5fxJQCjJ1w5WtmWigpGP2z, Hash160 Hamming: 29


Bitflip:  76%|███████▋  | 764897/1000000 [12:04<05:10, 756.01it/s]

New Best - Priv: 64cef70550c928c7244bc04befb3a713fed1850c34382db9a7df9ad6695b2847, Addr: 19BXkb67adspJi1agVZz3STY7vA9nNp8GU, Hash160 Hamming: 27


Bitflip: 100%|██████████| 1000000/1000000 [15:40<00:00, 1063.51it/s]


Results:
Best Priv: 64cef70550c928c7244bc04befb3a713fed1850c34382db9a7df9ad6695b2847
Best Addr: 19BXkb67adspJi1agVZz3STY7vA9nNp8GU
Best Hamming: 27


In [ ]:
!pip install pycryptodome ecdsa base58 tqdm

import hashlib
import binascii
from Crypto.Hash import SHA256, RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import random
from tqdm import tqdm

TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
TARGET_HASH160 = "a0b0d60e5991578ed37cbda2b17d8b2ce23ab295"
BASE_TIMESTAMP_START = 1298975179.002  # Feb 28, 2011, ~21:46 UTC
BASE_TIMESTAMP_END = 1298980279.999    # ~23:51 UTC
USER_ID = "895"
SALT = "e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36"
DELIMITER = "-"

def derive_address(seed, hash_type="single"):
    try:
        if hash_type == "single":
            priv = hashlib.sha256(seed.encode('utf-8')).hexdigest()
        elif hash_type == "sha256d":
            priv = hashlib.sha256(hashlib.sha256(seed.encode('utf-8')).digest()).hexdigest()
        else:  # concat
            priv = hashlib.sha256((seed + SALT).encode('utf-8')).hexdigest()

        priv_bytes = binascii.unhexlify(priv)
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = SHA256.new(pub).digest()
        ripemd = RIPEMD160.new(sha).digest()
        payload = b'\x00' + ripemd
        checksum = SHA256.new(SHA256.new(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        return addr, ripemd.hex(), priv
    except:
        return None, None, None

def hamming_distance(s1, s2):
    len_diff = abs(len(s1) - len(s2))
    return sum(c1 != c2 for c1, c2 in zip(s1, s2)) + len_diff

def generate_seed():
    # Components—fully randomized
    micros = random.uniform(0, (BASE_TIMESTAMP_END - BASE_TIMESTAMP_START) * 1000)  # Micros across 2 hours
    timestamp = f"{BASE_TIMESTAMP_START + micros / 1000:.6f}"
    iso_date = "20110228"
    jpy = f"{83.0 + random.uniform(-0.001, 0.1):.3f}"
    entropy = str(random.randint(4000, 5000))
    variant = str(random.randint(0, 5))

    # Prefixes and affixes—shuffled picks
    prefixes = [
        "mtgox", "mtgox_JPY", "tibanne", "karpeles",
        "KARPeles_mtgox_JPY", USER_ID
    ]
    affixes = [
        "", f"v{variant}", "v2011", "2011", "mtgox",
        "miku", "admin", "tibanne", SALT
    ]

    # All components in a bag—random order every time
    components = [
        timestamp, iso_date, jpy, entropy,
        random.choice(prefixes), random.choice(affixes), SALT
    ]
    random.shuffle(components)  # Chaos mode: structure order randomized

    # Join with delimiter, strip dupes while preserving order
    seed = DELIMITER.join(comp for comp in components if comp)
    return seed

def brute_force(sample_size=10000000):
    best_priv = None
    best_hamm_addr = float('inf')
    best_hamm_hash160 = float('inf')
    best_seed = None
    best_addr = None

    # Base check—no sequential bias
    base_seed = f"{USER_ID}{DELIMITER}mtgox{DELIMITER}1298975179.0028159{DELIMITER}{SALT}{DELIMITER}83.002{DELIMITER}4062{DELIMITER}2011"
    hash_types = ["single", "sha256d", "concat"]
    random.shuffle(hash_types)  # Randomize hash order too
    for hash_type in hash_types:
        base_addr, base_hash160, base_priv = derive_address(base_seed, hash_type)
        if base_addr:
            base_hamm_addr = hamming_distance(base_addr, TARGET_ADDRESS)
            base_hamm_hash160 = hamming_distance(base_hash160, TARGET_HASH160)
            print(f"Base - Seed: {base_seed}, Hash: {hash_type}, Addr: {base_addr}, Addr Hamming: {base_hamm_addr}, Hash160 Hamming: {base_hamm_hash160}")
            if base_hamm_hash160 < best_hamm_hash160:
                best_priv = base_priv
                best_hamm_addr = base_hamm_addr
                best_hamm_hash160 = base_hamm_hash160
                best_seed = base_seed
                best_addr = base_addr

    print(f"\nStarting brute-force with {sample_size} seeds...")
    seen_seeds = set()  # Track uniqueness
    for _ in tqdm(range(sample_size), desc="Chaos Brute-Forcing"):
        seed = generate_seed()
        while seed in seen_seeds:  # Force unique seeds
            seed = generate_seed()
        seen_seeds.add(seed)

        random.shuffle(hash_types)  # Random hash order per seed
        for hash_type in hash_types:
            addr, hash160, priv = derive_address(seed, hash_type)
            if addr is None:
                continue

            hamm_addr = hamming_distance(addr, TARGET_ADDRESS)
            hamm_hash160 = hamming_distance(hash160, TARGET_HASH160)
            if hamm_hash160 < best_hamm_hash160 or (hamm_hash160 == best_hamm_hash160 and hamm_addr < best_hamm_addr):
                best_priv = priv
                best_hamm_addr = hamm_addr
                best_hamm_hash160 = hamm_hash160
                best_seed = seed
                best_addr = addr
                print(f"New Best - Priv: {priv}, Seed: {seed}, Hash: {hash_type}, Addr: {addr}, Addr Hamming: {hamm_addr}, Hash160 Hamming: {hamm_hash160}")
            if hamm_hash160 == 0:
                print(f"Exact Match Found! Seed: {seed}, Hash: {hash_type}, Addr: {addr}")
                return priv, seed, 0, addr

    print(f"\nResults:")
    print(f"Best Priv: {best_priv}")
    print(f"Best Seed: {best_seed}")
    print(f"Best Addr: {best_addr}")
    print(f"Best Addr Hamming: {best_hamm_addr}")
    print(f"Best Hash160 Hamming: {best_hamm_hash160}")
    return best_priv, best_seed, best_hamm_addr, best_hamm_hash160, best_addr

# Full Chaos Send
random.seed()  # Fresh randomness each run
best_priv, best_seed, best_hamm_addr, best_hamm_hash160, best_addr = brute_force()

Base - Seed: 895-mtgox-1298975179.0028159-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-83.002-4062-2011, Hash: sha256d, Addr: 1PVEtxDJECWyiYRzdJ2c8RnUa4o7mnvajV, Addr Hamming: 33, Hash160 Hamming: 38
Base - Seed: 895-mtgox-1298975179.0028159-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-83.002-4062-2011, Hash: single, Addr: 1LW5NEdkCSDpXPNKV4QoE3LXumymqSXoig, Addr Hamming: 33, Hash160 Hamming: 39
Base - Seed: 895-mtgox-1298975179.0028159-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-83.002-4062-2011, Hash: concat, Addr: 1Avv9FERzB9hx4PUnzM7nbpDxdJLRmjQne, Addr Hamming: 32, Hash160 Hamming: 35

Starting brute-force with 10000000 seeds...


Chaos Brute-Forcing:   0%|          | 40/10000000 [00:00<7:00:39, 396.20it/s]

New Best - Priv: 102b4665e39e3ed8175707cd1d58c2ea1582399060d909f733513786856d9697, Seed: KARPeles_mtgox_JPY-20110228-tibanne-4017-1298979849.023898-83.055-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36, Hash: single, Addr: 13XiNN5ysdNx72dSxYSnqANo3eSCGdkArn, Addr Hamming: 33, Hash160 Hamming: 34
New Best - Priv: 8386e1272c73bfac6e7e35b9d3c48b2d9cf3c06c3cc8bb45ef05f4c27864b7f4, Seed: mtgox-83.050-1298976253.404829-4831-20110228-v2011-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36, Hash: concat, Addr: 17cNkk6Mm5fPiH8AV3jgfx3gshDTZcVKbq, Addr Hamming: 32, Hash160 Hamming: 34
New Best - Priv: ec2b41fd7f52fda447e1908b414f536afe7d75b21972c8022787099a00897cc3, Seed: mtgox_JPY-1298977184.126415-83.011-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-20110228-4511, Hash: concat, Addr: 1GPD44VAaFMNinivB1HbZbHV4cLppXUBfQ, Addr Hamming: 33, Hash160 Hamming: 33
New Best - Priv: e9e92c4d441001cd27332b27bc747eaa8e617b7bdbad29b5d1770ceec439b0d4, Seed: 1298978627.697399-4543-e67a0550848b79322f4b5e8

Chaos Brute-Forcing:   0%|          | 4777/10000000 [00:13<6:58:51, 397.72it/s]

New Best - Priv: ad10cd18af00cc185c57d7b8f7a36d80c912b9579ce219b950d8ccdf5a77793c, Seed: 4748-1298979176.623774-20110228-miku-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-tibanne-83.091, Hash: concat, Addr: 16m7n7ZhYo2ubKaTcnJTeBsNtYFjfkeza3, Addr Hamming: 33, Hash160 Hamming: 29


...

New Best - Priv: 615f91e6c5b114da9c399df22a8ad074926f8aec7f7f99d0897afcc30c140b5f, Seed: 4355-v2011-83.004-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-1298979992.295113-895-20110228, Hash: single, Addr: 1PVboT6EGPoNsrEqYCes66zd6kv6WmMNpL, Addr Hamming: 32, Hash160 Hamming: 28


Chaos Brute-Forcing:   2%|▏         | 227267/10000000 [10:33<6:49:45, 397.50it/s]

New Best - Priv: a333cb229e266ec30877d395774d4ccc06bd4b11a0c75c7168ff144ecbafd35b, Seed: 20110228-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-4175-1298977752.188775-83.004-v4-mtgox, Hash: single, Addr: 17jfAxL46PrqHcZhP15ks6YJFdCHNoTRS9, Addr Hamming: 32, Hash160 Hamming: 26


Chaos Brute-Forcing: 100%|██████████| 10000000/10000000 [7:48:29<00:00, 355.75it/s]



Results:
Best Priv: a333cb229e266ec30877d395774d4ccc06bd4b11a0c75c7168ff144ecbafd35b
Best Seed: 20110228-e67a0550848b79322f4b5e8b2c856ebfb6da9a26f664ca36-4175-1298977752.188775-83.004-v4-mtgox
Best Addr: 17jfAxL46PrqHcZhP15ks6YJFdCHNoTRS9
Best Addr Hamming: 32
Best Hash160 Hamming: 26


In [ ]:
!pip install pycryptodome ecdsa base58 tqdm

import binascii
from Crypto.Hash import SHA256, RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import random
from tqdm import tqdm

TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
TARGET_HASH160 = "a0b0d60e5991578ed37cbda2b17d8b2ce23ab295"
BASE_PRIVS = [
    "886e690e15699919897f879f8ce721b44af4cd7c918e261467ae7a38500fdf26",  # 28, 1FCDdBW… (new from 1M)
    "a333cb229e266ec30877d395774d4ccc06bd4b11a0c75c7168ff144ecbafd35b",  # 26, 17jfAxL… (best prior)
    # Add best from 6M run (~21:00 UTC) here—replace this comment
]

def privkey_to_address(priv_hex):
    try:
        priv_bytes = binascii.unhexlify(priv_hex)
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = SHA256.new(pub).digest()
        ripemd = RIPEMD160.new(sha).digest()
        payload = b'\x00' + ripemd
        checksum = SHA256.new(SHA256.new(payload).digest()).digest()[:4]
        return base58.b58encode(payload + checksum).decode(), ripemd.hex()
    except:
        return None, None

def hamming_distance(s1, s2):
    return sum(c1 != c2 for c1, c2 in zip(s1, s2)) + abs(len(s1) - len(s2))

def flip_bits(priv_hex, num_flips):
    priv_bytes = bytearray(binascii.unhexlify(priv_hex))
    positions = random.sample(range(256), num_flips)
    for pos in positions:
        byte_idx = pos // 8
        bit_idx = pos % 8
        priv_bytes[byte_idx] ^= (1 << bit_idx)
    return binascii.hexlify(priv_bytes).decode('utf-8')

def crunch_bitflip(base_privs, total_flips=6000000):  # 6M for ~3h
    flips_per_base = total_flips // len(base_privs)  # ~2M each
    best_priv = None
    best_addr = None
    best_hamm_hash160 = float('inf')

    for base_priv in base_privs:
        base_addr, base_hash160 = privkey_to_address(base_priv)
        base_hamm_hash160 = hamming_distance(base_hash160, TARGET_HASH160)
        print(f"Base - Priv: {base_priv}, Addr: {base_addr}, Hash160 Hamming: {base_hamm_hash160}")

        print(f"Starting {flips_per_base:,} flips for {base_priv[:8]}...")
        for _ in tqdm(range(flips_per_base), desc=f"Crunch Bitflip {base_priv[:8]}"):
            num_flips = random.randint(1, 32)
            flipped_priv = flip_bits(base_priv, num_flips)

            addr, hash160 = privkey_to_address(flipped_priv)
            if addr is None:
                continue

            hamm_hash160 = hamming_distance(hash160, TARGET_HASH160)
            if hamm_hash160 < best_hamm_hash160:
                best_priv = flipped_priv
                best_addr = addr
                best_hamm_hash160 = hamm_hash160
                print(f"New Best - Priv: {best_priv}, Addr: {best_addr}, Hash160 Hamming: {hamm_hash160}")
            if hamm_hash160 == 0:
                print(f"Exact Match! Priv: {best_priv}, Addr: {best_addr}")
                return best_priv, 0, best_addr

    print(f"\nResults (Second 6M Crunch):")
    print(f"Best Priv: {best_priv}")
    print(f"Best Addr: {best_addr}")
    print(f"Best Hash160 Hamming: {best_hamm_hash160}")
    return best_priv, best_hamm_hash160, best_addr

# Roll It (at ~21:00 UTC)
best_priv, best_hamm_hash160, best_addr = crunch_bitflip(BASE_PRIVS)

Base - Priv: 886e690e15699919897f879f8ce721b44af4cd7c918e261467ae7a38500fdf26, Addr: 1FCDdBWzY3x143UWt2P5mS42wcDRia2ZGx, Hash160 Hamming: 28
Starting 3,000,000 flips for 886e690e...


Crunch Bitflip 886e690e:   0%|          | 122/3000000 [00:00<41:15, 1211.97it/s]

New Best - Priv: 8c4f690e3568995de17b87dd8ce421f66af64d7d538e461067ae78385b0fdf26, Addr: 13pRFc4EYXC2Ln9PkdpEvrUsqgTYWc6srS, Hash160 Hamming: 40
New Best - Priv: 882e690fbd69b959897f839f8ce761b46af4cd7c910e063063ae7b38108f9f26, Addr: 1MdnL1J5MT7ASAVhThedHECvf3yXv1tyqB, Hash160 Hamming: 39
New Best - Priv: 846e690e1578991889ff871d8ce701b44af48d68a39a261467ae7a3c521fd702, Addr: 1CEEAxu5Vupqvg7frgStoNkXvojYQMXg7Z, Hash160 Hamming: 37
New Best - Priv: 886e694e156999198b3f87cf0de741a44af4cd6d918e261c67367a3c500f9f07, Addr: 12TbiUN3fpyZ33gsh5hXYh4hgBzS2PcsWB, Hash160 Hamming: 36
New Best - Priv: 886c690e1569b819895f869f8ee321f442e6ec3e91ce261423aefaba500ddf66, Addr: 1CK4W7fqb6DnJ9Sj2RVmddUuFvu3bXndd, Hash160 Hamming: 35
New Best - Priv: 884e6b0e15691919896f879f8ce721b45afccd78918e261463ae7a38500fda06, Addr: 1Gj6FcEKga74ncTdTfudkZFTYCuPk5qSpK, Hash160 Hamming: 33


Crunch Bitflip 886e690e:   0%|          | 480/3000000 [00:00<44:22, 1126.51it/s]

New Best - Priv: 886c690e15699919897f8f9f8ce721b44af0cd7c958e261467ae7a3c500fdf06, Addr: 19mNppHzgnncZ8Kx5aC81jb943JQBtZCzZ, Hash160 Hamming: 32


...

Crunch Bitflip 886e690e:  30%|███       | 908246/3000000 [14:38<32:50, 1061.70it/s]

New Best - Priv: 922c6b065569d917895f878f8ce629b46a74cd7c918f220466ae3a3a500cda06, Addr: 18MeLE1Zgovtdac7DJgPYuxQw2XsV8miKx, Hash160 Hamming: 27


Crunch Bitflip 886e690e: 100%|██████████| 3000000/3000000 [49:27<00:00, 1010.83it/s]


Base - Priv: a333cb229e266ec30877d395774d4ccc06bd4b11a0c75c7168ff144ecbafd35b, Addr: 17jfAxL46PrqHcZhP15ks6YJFdCHNoTRS9, Hash160 Hamming: 26
Starting 3,000,000 flips for a333cb22...


Crunch Bitflip a333cb22:  80%|███████▉  | 2396604/3000000 [40:29<09:42, 1036.29it/s]